# DF40 Dataset — Audit (Notebook 02)



# CELLULE 1 — MONTAGE DRIVE ET CONFIGURATION

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install tqdm pillow matplotlib pandas numpy scipy -q

import os
import hashlib
import random
import json
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm import tqdm
from scipy import stats

# Seed de reproductibilité
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Chemins
PROJECT_ROOT  = '/content/drive/MyDrive/Memoire_Deepfakes'
DATA_DIR      = f'{PROJECT_ROOT}/data'
RAW_DIR       = f'{DATA_DIR}/raw'
REAL_DIR      = f'{RAW_DIR}/DF40_real'
FAKE_BASE_DIR = f'{RAW_DIR}/DF40_fake_DMs'
MANIFEST_PATH = f'{DATA_DIR}/real_video_manifest.csv'
AUDIT_DIR     = f'{DATA_DIR}/audit'
os.makedirs(AUDIT_DIR, exist_ok=True)

# Paramètres attendus (v4)
EXPECTED_RESOLUTION = (256, 256)

DM_METHODS = ['MidJourney', 'ddim', 'DiT', 'SiT']   # CollabDiff exclu
DM_EXPECTED = {
    'MidJourney' : 1_594,   # 1600 - 5 corrompus - 1 ajustement ratio
    'ddim'       : 1_300,
    'DiT'        :   358,
    'SiT'        :   258,
}

# REAL_FF et REAL_CELEBDF ne sont plus vérifiés avec des valeurs fixes :
# le sous-échantillonnage vidéo-level aléatoire (seed=42) a retiré 132 vidéos
# réparties entre les deux sources de façon non-déterministe à ce stade.
# Seul le TOTAL est garanti.
EXPECTED_REAL_TOTAL     = 3_510
EXPECTED_FAKE_TOTAL     = 3_510
EXPECTED_GRAND_TOTAL    = 7_020

# Collecte des chemins
def get_image_paths(directory):
    """Retourne la liste triée de tous les chemins .jpg/.png d'un dossier."""
    exts = {'.jpg', '.jpeg', '.png'}
    paths = []
    for f in sorted(os.listdir(directory)):
        if Path(f).suffix.lower() in exts:
            paths.append(os.path.join(directory, f))
    return paths

real_paths = get_image_paths(REAL_DIR)
fake_paths = []
fake_paths_by_method = {}
for method in DM_METHODS:
    method_dir = os.path.join(FAKE_BASE_DIR, method)
    mpaths = get_image_paths(method_dir) if os.path.isdir(method_dir) else []
    fake_paths_by_method[method] = mpaths
    fake_paths.extend(mpaths)

print("=" * 70)
print("NOTEBOOK 02 — AUDIT DF40 v4")
print("=" * 70)
print(f"  REAL trouvés   : {len(real_paths):,}  (attendus : {EXPECTED_REAL_TOTAL:,})")
print(f"  FAKE trouvés   : {len(fake_paths):,}  (attendus : {EXPECTED_FAKE_TOTAL:,})")
print(f"  TOTAL          : {len(real_paths)+len(fake_paths):,}  (attendus : {EXPECTED_GRAND_TOTAL:,})")
print()
for method in DM_METHODS:
    found = len(fake_paths_by_method[method])
    expected = DM_EXPECTED[method]
    flag = '✅' if found == expected else '⚠️ ÉCART'
    print(f"  {flag}  {method:<15s} : {found:>5,} / {expected:>5,}")
print("=" * 70)
print("✅ Configuration chargée — Démarrage de l'audit")


# CELLULE 2 — D1 : INTÉGRITÉ PHYSIQUE

Vérifie pour chaque image : lisibilité, résolution 256×256, mode RGB.  
Les images défaillantes sont isolées dans un rapport.

In [ ]:
print("=" * 70)
print("D1 — INTÉGRITÉ PHYSIQUE")
print("=" * 70)

d1_issues = []

def audit_integrity(paths, label):
    """Vérifie l'intégrité physique d'une liste d'images."""
    n_ok = 0
    n_corrupt = 0
    n_wrong_res = 0
    n_wrong_mode = 0
    res_counter = defaultdict(int)

    for p in tqdm(paths, desc=f'D1 {label}', leave=True):
        try:
            img = Image.open(p)
            img.verify()            # Détecte les fichiers corrompus
            img = Image.open(p)     # Réouverture nécessaire après verify()
            w, h = img.size
            mode = img.mode
            res_counter[(w, h)] += 1

            issue = None
            if (w, h) != EXPECTED_RESOLUTION:
                n_wrong_res += 1
                issue = f'resolution_{w}x{h}'
            elif mode != 'RGB':
                n_wrong_mode += 1
                issue = f'mode_{mode}'
            else:
                n_ok += 1

            if issue:
                d1_issues.append({'file': os.path.basename(p), 'source': label, 'issue': issue})

        except Exception as e:
            n_corrupt += 1
            d1_issues.append({'file': os.path.basename(p), 'source': label, 'issue': f'corrupt: {e}'})

    return {
        'label': label, 'total': len(paths),
        'ok': n_ok, 'corrupt': n_corrupt,
        'wrong_resolution': n_wrong_res, 'wrong_mode': n_wrong_mode,
        'resolution_distribution': dict(res_counter)
    }

# Audit REAL
d1_real = audit_integrity(real_paths, 'REAL')

# Audit FAKE par méthode
d1_fake_by_method = {}
for method in DM_METHODS:
    d1_fake_by_method[method] = audit_integrity(fake_paths_by_method[method], f'FAKE/{method}')

# ── Résumé D1 ────────────────────────────────────────────────────────────────
print()
print(f"  {'Source':<20s} {'Total':>7} {'OK':>7} {'Corrompues':>12} {'Mauvaise rés.':>15} {'Mode≠RGB':>10}")
print("  " + "-" * 75)

def print_d1_row(r):
    flag = '✅' if r['corrupt'] == 0 and r['wrong_resolution'] == 0 and r['wrong_mode'] == 0 else '⚠️'
    print(f"  {flag} {r['label']:<18s} {r['total']:>7,} {r['ok']:>7,} "
          f"{r['corrupt']:>12,} {r['wrong_resolution']:>15,} {r['wrong_mode']:>10,}")

print_d1_row(d1_real)
for method in DM_METHODS:
    print_d1_row(d1_fake_by_method[method])

total_issues = len(d1_issues)
print()
if total_issues == 0:
    print("  ✅ D1 PASSED — Aucune anomalie d'intégrité détectée.")
else:
    print(f"  ⚠️  D1 WARNING — {total_issues} anomalie(s) détectée(s) :")
    for issue in d1_issues[:20]:
        print(f"      {issue}")
    if total_issues > 20:
        print(f"      ... et {total_issues - 20} autres (voir rapport D6)")

# Sauvegarde issues D1
d1_df = pd.DataFrame(d1_issues) if d1_issues else pd.DataFrame(columns=['file','source','issue'])
d1_df.to_csv(f'{AUDIT_DIR}/d1_integrity_issues.csv', index=False)
print(f"\n  Rapport D1 sauvegardé : audit/d1_integrity_issues.csv")


# CELLULE 3 — D2 : ÉQUILIBRE DES CLASSES ET DISTRIBUTION DM

Recomptage programmatique.

In [ ]:
print("=" * 70)
print("D2 — ÉQUILIBRE DES CLASSES")
print("=" * 70)

# Recomptage REAL par source
ff_imgs     = [p for p in real_paths if os.path.basename(p).startswith('FF_vid')]
celebdf_imgs = [p for p in real_paths if os.path.basename(p).startswith('CelebDF_vid')]

d2_real_counts = {
    'FF++'      : len(ff_imgs),
    'CelebDF-v2': len(celebdf_imgs),
    'TOTAL REAL': len(real_paths)
}

d2_fake_counts = {m: len(fake_paths_by_method[m]) for m in DM_METHODS}
d2_fake_counts['TOTAL FAKE'] = len(fake_paths)

print("\n  REAL :")
for k, v in d2_real_counts.items():
    pct = v / len(real_paths) * 100 if k != 'TOTAL REAL' else 100.0
    exp_map = {'FF++': EXPECTED_REAL_FF, 'CelebDF-v2': EXPECTED_REAL_CELEBDF, 'TOTAL REAL': EXPECTED_REAL_TOTAL}
    exp = exp_map[k]
    flag = '✅' if v == exp else '⚠️'
    print(f"    {flag} {k:<15s} : {v:>5,}  (attendu : {exp:,})  [{pct:.1f}%]")

print("\n  FAKE :")
for method in DM_METHODS:
    v = d2_fake_counts[method]
    exp = DM_EXPECTED[method]
    pct = v / len(fake_paths) * 100
    flag = '✅' if v == exp else '⚠️'
    print(f"    {flag} {method:<15s} : {v:>5,}  (attendu : {exp:,})  [{pct:.1f}%]")
v_tot = d2_fake_counts['TOTAL FAKE']
flag = '✅' if v_tot == EXPECTED_FAKE_TOTAL else '⚠️'
print(f"    {flag} {'TOTAL FAKE':<15s} : {v_tot:>5,}  (attendu : {EXPECTED_FAKE_TOTAL:,})")

# Ratio global
ratio = len(real_paths) / max(len(fake_paths), 1)
balance_flag = '✅' if abs(ratio - 1.0) < 0.01 else '⚠️'
print(f"\n  {balance_flag} Ratio REAL/FAKE : {ratio:.4f}  (idéal : 1.0000)")

# Graphique D2
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('D2 — Distribution des classes', fontsize=13, fontweight='bold')

# Pie REAL vs FAKE
axes[0].pie(
    [len(real_paths), len(fake_paths)],
    labels=[f'REAL\n{len(real_paths):,}', f'FAKE\n{len(fake_paths):,}'],
    autopct='%1.1f%%', startangle=90,
    colors=['#4CAF50', '#F44336'],
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Équilibre global REAL / FAKE', fontweight='bold')

# Barres méthodes DM
methods = list(DM_METHODS)
found_vals = [d2_fake_counts[m] for m in methods]
expected_vals = [DM_EXPECTED[m] for m in methods]
x = np.arange(len(methods))
width = 0.35
bars1 = axes[1].bar(x - width/2, found_vals, width, label='Trouvé', color='#2196F3', alpha=0.85)
bars2 = axes[1].bar(x + width/2, expected_vals, width, label='Attendu', color='#FF9800', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, rotation=15, ha='right')
axes[1].set_ylabel('Nombre d\'images')
axes[1].set_title('Distribution FAKE par méthode DM', fontweight='bold')
axes[1].legend()
axes[1].bar_label(bars1, padding=3, fmt='%d', fontsize=8)

plt.tight_layout()
fig.savefig(f'{AUDIT_DIR}/d2_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Graphique sauvegardé : data/audit/d2_class_balance.png")


# CELLULE 4 — D3 : DISTRIBUTIONS STATISTIQUES DES PIXELS

Cal calcule luminosité (L), contraste (σ) et saturation (S) sur un échantillon.  
Test Mann-Whitney U : REAL vs FAKE sur chaque métrique.  

In [ ]:
print("=" * 70)
print("D3 — DISTRIBUTIONS STATISTIQUES DES PIXELS")
print("=" * 70)

SAMPLE_SIZE = 400   # Images par classe pour les stats (équilibre vitesse/précision)

def extract_pixel_stats(paths, sample_size, label):
    """Extrait luminosité moyenne, contraste et saturation d'un échantillon d'images."""
    sample = random.sample(paths, min(sample_size, len(paths)))
    lum, contrast, sat = [], [], []
    for p in tqdm(sample, desc=f'D3 stats {label}', leave=True):
        try:
            img = Image.open(p).convert('RGB')
            arr = np.array(img, dtype=np.float32)
            gray = 0.299 * arr[:,:,0] + 0.587 * arr[:,:,1] + 0.114 * arr[:,:,2]
            lum.append(float(gray.mean()))
            contrast.append(float(gray.std()))
            # Saturation via HSV
            img_hsv = img.convert('HSV') if hasattr(Image, 'HSV') else img
            r, g, b = arr[:,:,0]/255, arr[:,:,1]/255, arr[:,:,2]/255
            max_c = np.maximum(np.maximum(r, g), b)
            min_c = np.minimum(np.minimum(r, g), b)
            s = np.where(max_c > 0, (max_c - min_c) / max_c, 0)
            sat.append(float(s.mean()))
        except Exception:
            pass
    return np.array(lum), np.array(contrast), np.array(sat)

real_sample = random.sample(real_paths, min(SAMPLE_SIZE, len(real_paths)))
fake_sample = random.sample(fake_paths,  min(SAMPLE_SIZE, len(fake_paths)))

lum_r, con_r, sat_r = extract_pixel_stats(real_sample, SAMPLE_SIZE, 'REAL')
lum_f, con_f, sat_f = extract_pixel_stats(fake_sample,  SAMPLE_SIZE, 'FAKE')

# ── Tests Mann-Whitney U ──────────────────────────────────────────────────────
def mwu_test(a, b, name):
    stat, pval = stats.mannwhitneyu(a, b, alternative='two-sided')
    sig = 'p<0.05 ✅' if pval < 0.05 else 'p≥0.05 ⚠️'
    print(f"    {name:<20s} | REAL μ={a.mean():.2f} σ={a.std():.2f} "
          f"| FAKE μ={b.mean():.2f} σ={b.std():.2f} "
          f"| U={stat:.0f}, p={pval:.4f}  [{sig}]")
    return {'metric': name, 'real_mean': float(a.mean()), 'real_std': float(a.std()),
            'fake_mean': float(b.mean()), 'fake_std': float(b.std()),
            'U_stat': float(stat), 'p_value': float(pval), 'significant': pval < 0.05}

print(f"\n  Échantillon : {len(lum_r)} REAL + {len(lum_f)} FAKE")
print(f"  {'Métrique':<20s}   {'REAL':^28s}   {'FAKE':^28s}   {'Test MW-U'}")
print("  " + "-" * 95)
d3_results = [
    mwu_test(lum_r, lum_f, 'Luminosité'),
    mwu_test(con_r, con_f, 'Contraste'),
    mwu_test(sat_r, sat_f, 'Saturation'),
]
d3_df = pd.DataFrame(d3_results)
d3_df.to_csv(f'{AUDIT_DIR}/d3_pixel_statistics.csv', index=False)

# ── Graphiques D3 ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('D3 — Distributions pixel REAL vs FAKE', fontsize=13, fontweight='bold')
metrics = [
    ('Luminosité (0-255)', lum_r, lum_f),
    ('Contraste (σ)', con_r, con_f),
    ('Saturation (0-1)', sat_r, sat_f),
]
for ax, (title, r_vals, f_vals) in zip(axes, metrics):
    ax.hist(r_vals, bins=40, alpha=0.6, color='#4CAF50', label='REAL', density=True)
    ax.hist(f_vals, bins=40, alpha=0.6, color='#F44336', label='FAKE', density=True)
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.set_xlabel(title)
    ax.set_ylabel('Densité')

plt.tight_layout()
fig.savefig(f'{AUDIT_DIR}/d3_pixel_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n  Graphique sauvegardé : data/audit/d3_pixel_distributions.png")
print(f"  CSV sauvegardé       : data/audit/d3_pixel_statistics.csv")


# CELLULE 5 — D4 : DÉTECTION DE DOUBLONS MD5

Calcul de l'empreinte MD5 de chaque image.  
- Doublons **intra-classe** (REAL-REAL ou FAKE-FAKE) : redondance acceptable mais signalée  
- Doublons **inter-classes** (même image classée REAL et FAKE) : erreur méthodologique grave  

In [ ]:
print("=" * 70)
print("D4 — DÉTECTION DE DOUBLONS MD5")
print("=" * 70)
print("  ⏳ Calcul MD5 en cours — peut prendre 5-10 minutes...")

def compute_md5(path):
    """Calcule l'empreinte MD5 d'un fichier image."""
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

def build_md5_dict(paths, label):
    md5_map = {}   # md5 → [paths]
    for p in tqdm(paths, desc=f'MD5 {label}', leave=True):
        try:
            digest = compute_md5(p)
            if digest not in md5_map:
                md5_map[digest] = []
            md5_map[digest].append(p)
        except Exception as e:
            print(f"    ⚠️  Erreur MD5 sur {os.path.basename(p)} : {e}")
    return md5_map

md5_real = build_md5_dict(real_paths, 'REAL')
md5_fake = build_md5_dict(fake_paths, 'FAKE')

# ── Doublons intra-classe ─────────────────────────────────────────────────────
dup_real_intra = {k: v for k, v in md5_real.items() if len(v) > 1}
dup_fake_intra = {k: v for k, v in md5_fake.items() if len(v) > 1}

# ── Doublons inter-classes ────────────────────────────────────────────────────
real_hashes = set(md5_real.keys())
fake_hashes = set(md5_fake.keys())
cross_class_hashes = real_hashes & fake_hashes

cross_class_dupes = []
for h in cross_class_hashes:
    for rp in md5_real[h]:
        for fp in md5_fake[h]:
            cross_class_dupes.append({
                'md5': h,
                'real_file': os.path.basename(rp),
                'fake_file': os.path.basename(fp)
            })

# ── Affichage résultats D4 ────────────────────────────────────────────────────
print(f"\n  Doublons intra-classe REAL  : {sum(len(v)-1 for v in dup_real_intra.values()):,} "
      f"(sur {len(dup_real_intra)} empreintes dupliquées)")
flag_real = '✅' if not dup_real_intra else '⚠️'
print(f"  {flag_real}  Groupes dupliqués REAL : {len(dup_real_intra)}")

print(f"\n  Doublons intra-classe FAKE  : {sum(len(v)-1 for v in dup_fake_intra.values()):,} "
      f"(sur {len(dup_fake_intra)} empreintes dupliquées)")
flag_fake = '✅' if not dup_fake_intra else '⚠️'
print(f"  {flag_fake}  Groupes dupliqués FAKE : {len(dup_fake_intra)}")

flag_cross = '✅' if not cross_class_dupes else '🚨 ERREUR CRITIQUE'
print(f"\n  {flag_cross}  Doublons inter-classes (REAL ∩ FAKE) : {len(cross_class_dupes)}")
if cross_class_dupes:
    print("\n  🚨 LISTE DES DOUBLONS INTER-CLASSES :")
    for d in cross_class_dupes[:10]:
        print(f"      MD5={d['md5'][:12]}... | REAL={d['real_file']} | FAKE={d['fake_file']}")

# ── Sauvegarde rapports D4 ────────────────────────────────────────────────────
pd.DataFrame(cross_class_dupes).to_csv(f'{AUDIT_DIR}/d4_cross_class_duplicates.csv', index=False)

intra_rows = []
for h, paths_list in {**dup_real_intra, **dup_fake_intra}.items():
    for p in paths_list:
        intra_rows.append({'md5': h, 'file': os.path.basename(p)})
pd.DataFrame(intra_rows).to_csv(f'{AUDIT_DIR}/d4_intra_class_duplicates.csv', index=False)

print(f"\n  CSV inter-classes : audit/d4_cross_class_duplicates.csv")
print(f"  CSV intra-classes : audit/d4_intra_class_duplicates.csv")

# Résultat final D4
d4_passed = (len(cross_class_dupes) == 0)
if d4_passed:
    print("\n  ✅ D4 PASSED — Aucun doublon inter-classes détecté.")
else:
    print(f"\n  🚨 D4 FAILED — {len(cross_class_dupes)} doublon(s) inter-classes. Action requise avant de continuer.")


# CELLULE 6 — D5 : INSPECTION VISUELLE STRUCTURÉE

Génère des grilles d'images aléatoires (5 images × N sources).  
Objectif : valider visuellement que les visages sont correctement extraits, cadrés, sans artefact d'extraction visible (bandes noires, troncatures, etc.).

In [ ]:
print("=" * 70)
print("D5 — INSPECTION VISUELLE STRUCTURÉE")
print("=" * 70)

N_SAMPLES_VIZ = 8   # Images par ligne

def load_img_array(path):
    """Charge une image en tableau numpy RGB."""
    try:
        return np.array(Image.open(path).convert('RGB'))
    except Exception:
        return np.zeros((256, 256, 3), dtype=np.uint8)

# ── Grille REAL par source ────────────────────────────────────────────────────
real_sources = {
    'FF++ (REAL)'      : [p for p in real_paths if os.path.basename(p).startswith('FF_vid')],
    'CelebDF-v2 (REAL)': [p for p in real_paths if os.path.basename(p).startswith('CelebDF_vid')],
}

fig, axes = plt.subplots(len(real_sources), N_SAMPLES_VIZ,
                          figsize=(N_SAMPLES_VIZ * 2.2, len(real_sources) * 2.5))
fig.suptitle('D5 — Inspection visuelle : images REAL', fontsize=13, fontweight='bold')

for row_idx, (src_name, src_paths) in enumerate(real_sources.items()):
    sample = random.sample(src_paths, min(N_SAMPLES_VIZ, len(src_paths)))
    for col_idx in range(N_SAMPLES_VIZ):
        ax = axes[row_idx, col_idx] if len(real_sources) > 1 else axes[col_idx]
        ax.imshow(load_img_array(sample[col_idx]) if col_idx < len(sample) else np.zeros((256,256,3), dtype=np.uint8))
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(src_name, fontsize=9, rotation=0, labelpad=70, va='center')

plt.tight_layout()
fig.savefig(f'{AUDIT_DIR}/d5_visual_real.png', dpi=120, bbox_inches='tight')
plt.show()
print("  Grille REAL sauvegardée : audit/d5_visual_real.png")

# ── Grille FAKE par méthode DM ────────────────────────────────────────────────
fig, axes = plt.subplots(len(DM_METHODS), N_SAMPLES_VIZ,
                          figsize=(N_SAMPLES_VIZ * 2.2, len(DM_METHODS) * 2.5))
fig.suptitle('D5 — Inspection visuelle : images FAKE par méthode DM', fontsize=13, fontweight='bold')

for row_idx, method in enumerate(DM_METHODS):
    mpaths = fake_paths_by_method[method]
    sample = random.sample(mpaths, min(N_SAMPLES_VIZ, len(mpaths)))
    for col_idx in range(N_SAMPLES_VIZ):
        ax = axes[row_idx, col_idx]
        ax.imshow(load_img_array(sample[col_idx]) if col_idx < len(sample) else np.zeros((256,256,3), dtype=np.uint8))
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(method, fontsize=9, rotation=0, labelpad=65, va='center')

plt.tight_layout()
fig.savefig(f'{AUDIT_DIR}/d5_visual_fake.png', dpi=120, bbox_inches='tight')
plt.show()
print("  Grille FAKE sauvegardée : audit/d5_visual_fake.png")
print("\n  ✅ D5 — Inspectez visuellement les grilles ci-dessus.")
print("     Points de contrôle :")
print("       □ Visages bien centrés et recadrés")
print("       □ Pas de bandes noires / images tronquées")
print("       □ Résolution visiblement uniforme")
print("       □ Pas d'images aberrantes (fond uni, artefacts majeurs)")
print("       □ Diversité satisfaisante des identités et expressions")


# CELLULE 7 — D6 : RAPPORT D'AUDIT FORMEL

Génère un rapport complet.
Ce fichier constitue la **pièce justificative officielle** de l'audit du dataset.

In [ ]:
print("=" * 70)
print("D6 — RAPPORT D'AUDIT FORMEL")
print("=" * 70)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
report_path = f'{AUDIT_DIR}/audit_report_v4_{ts}.txt'

def d1_summary():
    total_issues_real = d1_real['corrupt'] + d1_real['wrong_resolution'] + d1_real['wrong_mode']
    total_issues_fake = sum(
        r['corrupt'] + r['wrong_resolution'] + r['wrong_mode']
        for r in d1_fake_by_method.values()
    )
    return total_issues_real + total_issues_fake

d1_total_issues   = d1_summary()
d2_ratio_ok       = abs(len(real_paths) / max(len(fake_paths), 1) - 1.0) < 0.01
d3_sig_count      = sum(1 for r in d3_results if r['significant'])
d4_cross_ok       = (len(cross_class_dupes) == 0)
d4_intra_real_ok  = (len(dup_real_intra) == 0)
d4_intra_fake_ok  = (len(dup_fake_intra) == 0)

overall_passed = (d1_total_issues == 0 and d2_ratio_ok and d4_cross_ok)

with open(report_path, 'w', encoding='utf-8') as f:
    sep = '=' * 70
    f.write(sep + '\n')
    f.write("RAPPORT D'AUDIT — DATASET DF40 v4\n")
    f.write("Mémoire : Obsolescence des détecteurs de deepfakes\n")
    f.write("Auteur  : Maxime Ducarme\n")
    f.write(sep + '\n')
    f.write(f"Date         : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Dataset      : DF40 (DeepfakeBench v2)\n")
    f.write(f"Version      : v4 (split vidéo-level)\n")
    f.write(f"Seed         : {RANDOM_SEED}\n")
    f.write(f"Verdict      : {'✅ PASSED' if overall_passed else '⚠️  ISSUES DETECTED'}\n\n")

    f.write("D1 — INTÉGRITÉ PHYSIQUE\n")
    f.write('-' * 70 + '\n')
    f.write(f"  Images REAL    : {d1_real['total']:,} | OK={d1_real['ok']:,} | Corrompues={d1_real['corrupt']} | Mauvaise rés.={d1_real['wrong_resolution']} | Mode≠RGB={d1_real['wrong_mode']}\n")
    for m in DM_METHODS:
        r = d1_fake_by_method[m]
        f.write(f"  FAKE/{m:<13s}: {r['total']:,} | OK={r['ok']:,} | Corrompues={r['corrupt']} | Mauvaise rés.={r['wrong_resolution']} | Mode≠RGB={r['wrong_mode']}\n")
    f.write(f"  Résultat : {'✅ PASSED' if d1_total_issues == 0 else f'⚠️  {d1_total_issues} anomalie(s)'}\n\n")

    f.write("D2 — ÉQUILIBRE DES CLASSES\n")
    f.write('-' * 70 + '\n')
    f.write(f"  REAL total     : {len(real_paths):,} (attendu : {EXPECTED_REAL_TOTAL:,})\n")
    f.write(f"    FF++         : {len(ff_imgs):,} (attendu : {EXPECTED_REAL_FF:,})\n")
    f.write(f"    CelebDF-v2   : {len(celebdf_imgs):,} (attendu : {EXPECTED_REAL_CELEBDF:,})\n")
    f.write(f"  FAKE total     : {len(fake_paths):,} (attendu : {EXPECTED_FAKE_TOTAL:,})\n")
    for m in DM_METHODS:
        f.write(f"    {m:<15s}: {len(fake_paths_by_method[m]):,} (attendu : {DM_EXPECTED[m]:,})\n")
    f.write(f"  Ratio REAL/FAKE: {len(real_paths)/max(len(fake_paths),1):.4f}\n")
    f.write(f"  Résultat : {'✅ PASSED' if d2_ratio_ok else '⚠️  Ratio déséquilibré'}\n\n")

    f.write("D3 — DISTRIBUTIONS STATISTIQUES (Mann-Whitney U)\n")
    f.write('-' * 70 + '\n')
    f.write(f"  Échantillon : {len(lum_r)} REAL + {len(lum_f)} FAKE\n")
    for r in d3_results:
        sig_str = 'p<0.05 ✅' if r['significant'] else 'p≥0.05 ⚠️'
        f.write(f"  {r['metric']:<20s}: REAL μ={r['real_mean']:.2f}, FAKE μ={r['fake_mean']:.2f} "
                f"| p={r['p_value']:.4f} [{sig_str}]\n")
    f.write(f"  {d3_sig_count}/3 métriques significativement différentes\n\n")

    f.write("D4 — DOUBLONS MD5\n")
    f.write('-' * 70 + '\n')
    f.write(f"  Doublons intra REAL : {len(dup_real_intra)} groupe(s)  {'✅' if d4_intra_real_ok else '⚠️'}\n")
    f.write(f"  Doublons intra FAKE : {len(dup_fake_intra)} groupe(s)  {'✅' if d4_intra_fake_ok else '⚠️'}\n")
    f.write(f"  Doublons inter-classes (REAL ∩ FAKE) : {len(cross_class_dupes)}  {'✅' if d4_cross_ok else '🚨 CRITIQUE'}\n")
    f.write(f"  Résultat : {'✅ PASSED' if d4_cross_ok else '🚨 FAILED — Action requise'}\n\n")

    f.write("D5 — INSPECTION VISUELLE\n")
    f.write('-' * 70 + '\n')
    f.write("  Grilles générées — validation humaine requise.\n")
    f.write("  Fichiers : audit/d5_visual_real.png, audit/d5_visual_fake.png\n\n")

    f.write("VERDICT GLOBAL\n")
    f.write('-' * 70 + '\n')
    f.write(f"  D1 Intégrité        : {'✅ PASSED' if d1_total_issues == 0 else f'⚠️  {d1_total_issues} issue(s)'}\n")
    f.write(f"  D2 Équilibre        : {'✅ PASSED' if d2_ratio_ok else '⚠️  Déséquilibre'}\n")
    f.write(f"  D3 Stats pixels     : {d3_sig_count}/3 métriques significatives\n")
    f.write(f"  D4 Doublons         : {'✅ PASSED' if d4_cross_ok else '🚨 FAILED'}\n")
    f.write(f"  D5 Visuel           : Validation manuelle requise\n")
    f.write(f"  \n")
    f.write(f"  RÉSULTAT GLOBAL     : {'✅ DATASET CERTIFIÉ — Procéder au notebook 03' if overall_passed else '⚠️  ANOMALIES DÉTECTÉES — Voir détails ci-dessus'}\n")
    f.write(sep + '\n')

print(f"  Rapport complet sauvegardé : {report_path}\n")
with open(report_path, encoding='utf-8') as f:
    print(f.read())


# CELLULE 8 — RÉSUMÉ FINAL ET PROCHAINES ÉTAPES

Cette cellule affiche un tableau récapitulatif lisible et rappelle les actions à effectuer.

In [ ]:
print("=" * 70)
print("NOTEBOOK 02 — RÉSUMÉ D'AUDIT")
print("=" * 70)
print()
print(f"  {'Dimension':<30s} {'Statut':<30s}")
print("  " + "-" * 60)
print(f"  {'D1 — Intégrité physique':<30s} {'✅ PASSED' if d1_total_issues == 0 else f'⚠️  {d1_total_issues} anomalie(s)'}")
print(f"  {'D2 — Équilibre classes':<30s} {'✅ PASSED' if d2_ratio_ok else '⚠️  Déséquilibre détecté'}")
print(f"  {'D3 — Distributions pixel':<30s} {d3_sig_count}/3 métriques significativement diff.")
print(f"  {'D4 — Doublons MD5':<30s} {'✅ 0 doublon inter-classes' if d4_cross_ok else '🚨 DOUBLONS INTER-CLASSES'}")
print(f"  {'D5 — Inspection visuelle':<30s} Validation manuelle requise")
print(f"  {'D6 — Rapport formel':<30s} ✅ Généré dans audit/")
print()
print("  Fichiers générés dans Memoire_Deepfakes/data/audit/ :")
for fname in sorted(os.listdir(AUDIT_DIR)):
    fsize = os.path.getsize(os.path.join(AUDIT_DIR, fname))
    print(f"    {fname:<50s} {fsize/1024:.1f} KB")
print()
if overall_passed:
    print("  ✅ DATASET CERTIFIÉ — Vous pouvez procéder au notebook 03 (split vidéo-level).")
else:
    print("  ⚠️  ANOMALIES DÉTECTÉES — Consultez le rapport d'audit avant de continuer.")
print()
print("  PROCHAINES ÉTAPES :")
print("    [1] Valider visuellement d5_visual_real.png et d5_visual_fake.png")
print("    [2] Consulter audit_report_v4_*.txt pour le rapport complet")
print("    [3] Créer et exécuter 03_split_data.ipynb (split vidéo-level 20/40/40)")
print("    [4] Tester le chargement des 4 poids .pth en Colab")
print("="*70)
